# Notebook 02 - Datenaufbereitung

- Datenbereinigung (Fehlende Werte, Ausreisser)
- Loops & Kontrollstrukturen
- Pandas DataFrames
- LLM-Unterstuetzung via Claude API (Bonuspunkt)

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, re, json, requests
from pathlib import Path
pd.set_option('display.float_format', '{:.1f}'.format)
print('Imports OK')

## 1. Rohdaten laden

In [ ]:
df = pd.read_csv('../data/inserate_roh.csv')
print(f'Rohdaten: {df.shape[0]} x {df.shape[1]}')
print('Fehlende Werte:'); print(df.isnull().sum())
df.head()

## 2. Datenbereinigung mit Loops & Kontrollstrukturen

In [ ]:
print(f'Start: {len(df)} Zeilen')
# Schritt 1: Fehlende Werte entfernen
df = df.dropna(subset=['preis_chf','flaeche_m2','zimmer_anzahl','stadt'])
print(f'Nach dropna: {len(df)} Zeilen')
# Schritt 2: Plausibilitaets-Filter mit Loop + Conditionals
ausgeschlossen, behalten = [], []
for idx, zeile in df.iterrows():
    grund = None
    if not (300 <= zeile['preis_chf'] <= 15000):
        grund = f"Preis: {zeile['preis_chf']:.0f}"
    elif not (15 <= zeile['flaeche_m2'] <= 400):
        grund = f"Flaeche: {zeile['flaeche_m2']:.0f}"
    elif not (0.5 <= zeile['zimmer_anzahl'] <= 12):
        grund = f"Zimmer: {zeile['zimmer_anzahl']}"
    if grund: ausgeschlossen.append({'idx':idx,'grund':grund})
    else: behalten.append(idx)
df = df.loc[behalten].copy()
print(f'Nach Filter: {len(df)} Zeilen, {len(ausgeschlossen)} ausgeschlossen')
for a in ausgeschlossen[:3]: print(f'  -> {a["grund"]}')

In [ ]:
# Features ableiten
df['preis_pro_m2'] = (df['preis_chf'] / df['flaeche_m2']).round(2)
def preiskategorie(p): return 'guenstig' if p<1200 else 'mittel' if p<2200 else 'teuer'
def zimmer_gruppe(z): return '1-1.5 Zi' if z<=1.5 else '2-2.5 Zi' if z<=2.5 else '3-3.5 Zi' if z<=3.5 else '4+ Zi'
df['preiskategorie'] = df['preis_chf'].apply(preiskategorie)
df['zimmer_gruppe'] = df['zimmer_anzahl'].apply(zimmer_gruppe)
print(df[['preis_chf','flaeche_m2','preis_pro_m2','preiskategorie','zimmer_gruppe']].head(6))

## 3. LLM-Analyse mit Claude API (Bonuspunkt)
Nutzt die Anthropic Claude API um Wohnungsbeschreibungen zu klassifizieren.

In [ ]:
def llm_extrahiere_ausstattung(beschreibungen):
    """Claude API: Extrahiert Ausstattungsmerkmale aus Beschreibungen."""
    batch = '\n---\n'.join([f'[{i}] {d}' for i,d in enumerate(beschreibungen)])
    prompt = f"""Analysiere {len(beschreibungen)} Wohnungsbeschreibungen.
Antworte NUR mit JSON-Array, Felder: balkon, parking, renoviert, luxus (boolean).
Beschreibungen:\n{batch}\nJSON:"""
    try:
        r = requests.post('https://api.anthropic.com/v1/messages',
            headers={'Content-Type':'application/json'},
            json={'model':'claude-sonnet-4-20250514','max_tokens':500,
                  'messages':[{'role':'user','content':prompt}]}, timeout=30)
        r.raise_for_status()
        text = r.json()['content'][0]['text'].strip()
        m = re.search(r'\[.*\]', text, re.DOTALL)
        if m: return json.loads(m.group())
    except Exception as e: print(f'LLM-Fehler: {e}')
    return [{'balkon':False,'parking':False,'renoviert':False,'luxus':False}]*len(beschreibungen)

print('LLM-Analyse...')
df_b = df[df['beschreibung'].notna() & (df['beschreibung']!='')].copy()
if len(df_b)>0:
    ergebnisse = []
    for i in range(0, min(len(df_b),30), 10):
        batch = df_b['beschreibung'].iloc[i:i+10].tolist()
        ergebnisse.extend(llm_extrahiere_ausstattung(batch))
        print(f'  Batch {i//10+1}: {len(batch)} analysiert')
    adf = pd.DataFrame(ergebnisse, index=df_b.index[:len(ergebnisse)])
    df = df.join(adf, how='left')
    for c in ['balkon','parking','renoviert','luxus']: df[c] = df[c].fillna(False)
    print('LLM-Features hinzugefuegt:'); print(df[['balkon','parking','renoviert','luxus']].sum())
else:
    print('Keine Beschreibungen -> LLM uebersprungen')
    for c in ['balkon','parking','renoviert','luxus']: df[c] = False

## 4. Bereinigten Datensatz speichern

In [ ]:
df.to_csv('../data/inserate_bereinigt.csv', index=False)
print(f'Gespeichert: {len(df)} x {len(df.columns)}')
print(df['preiskategorie'].value_counts())